#### <b>Import Library</b>

In [6]:
import numpy as np
import pandas as pd
import seaborn as sns
import os


#### <b>Load dataset</b>

In [7]:
file_lists= os.listdir("ecg_chunks")

In [8]:
len(file_lists)

131

#### <b> Create numpy array list</b>

In [9]:
data = np.concatenate(
    [np.load(f"ecg_chunks/{file}").astype(np.float32) for file in file_lists[:50]],
    axis=0
)

#### <b>X - <i>Independent Variable</i></b>

In [12]:
X = data
X.shape

(50000, 5000, 12)

#### <b>Y - Dedepandent Variable</b>

In [11]:
df = pd.read_csv("C:/Users/ytsub/Downloads/dataset/mimic-iv-ecg-diagnostic-electrocardiogram-matched-subset-1.0/mimic-iv-ecg-diagnostic-electrocardiogram-matched-subset-1.0/machine_measurements.csv")

C:\Users\ytsub\AppData\Local\Temp\ipykernel_23528\698130725.py:1: DtypeWarning: Columns (16,17,18,19,20,21) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("C:/Users/ytsub/Downloads/dataset/mimic-iv-ecg-diagnostic-electrocardiogram-matched-subset-1.0/mimic-iv-ecg-diagnostic-electrocardiogram-matched-subset-1.0/machine_measurements.csv")


In [13]:
report_cols = [f"report_{i}" for i in range(6)]
df['full_report'] = df[report_cols].fillna('').agg(' '.join, axis=1)

In [14]:
keywords = [
    "MI",
    "INFARCT",
    "ST ELEVATION",
    "ISCHEMIA"
]

def get_label(text):
    text = text.upper()
    for kw in keywords:
        if kw in text:
            return 1   # Heart attack / abnormal
    return 0           # Normal

df['label'] = df['full_report'].apply(get_label)

y = df['label'].values

In [16]:
y.shape
y = y[:50000]

In [18]:
Y = pd.DataFrame(y, columns=["traget"])
Y.shape

(50000, 1)

In [25]:
Y.value_counts()

traget
0         32436
1         17564
Name: count, dtype: int64

In [19]:
import tensorflow as tf
from tensorflow import keras


In [33]:
model = keras.Sequential([
    keras.layers.Conv1D(filters=32,kernel_size=5,activation='relu',input_shape=(5000,12)),
    keras.layers.MaxPooling1D(pool_size=2),
    keras.layers.Conv1D(filters=64,kernel_size=5,activation='relu'),
    keras.layers.MaxPooling1D(pool_size=2),
    keras.layers.Conv1D(filters=128,kernel_size=3,activation='relu'),
    keras.layers.MaxPooling1D(pool_size=2),
    keras.layers.Flatten(),
    keras.layers.Dense(128, activation="relu"),
    keras.layers.Dropout(0.5),
    keras.layers.Dense(1, activation="sigmoid")
])

c:\Users\ytsub\Desktop\github\Know_Your_Health_MLModel\myenv\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [34]:
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [35]:
history = model.fit(
    X,
    Y,
    epochs=30,
    batch_size=32,
    validation_split=0.2
)

MemoryError: Unable to allocate 8.94 GiB for an array with shape (40000, 5000, 12) and data type float32